In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_PATH = Path("../data/processed/clean_transactions.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Number of customers:", df["CustomerID"].nunique())

Dataset shape: (391298, 8)
Number of customers: 4335


In [3]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print(df["InvoiceDate"].dtype)
print(df["InvoiceDate"].min())
print(df["InvoiceDate"].max())

datetime64[us]
2010-12-01 08:26:00
2011-12-09 12:50:00


In [4]:
df["TransactionValue"] = (
    df["Quantity"] * df["UnitPrice"]
)

df[
    [
        "InvoiceNo",
        "CustomerID",
        "Quantity",
        "UnitPrice",
        "TransactionValue"
    ]
].head()

,InvoiceNo,CustomerID,Quantity,UnitPrice,TransactionValue
0,536365,17850.0,6,2.55,15.30
1,536365,17850.0,6,3.39,20.34
2,536365,17850.0,8,2.75,22.00
3,536365,17850.0,6,3.39,20.34
4,536365,17850.0,6,3.39,20.34


In [5]:
# Step 1: Define reference date

reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

print("Latest transaction:", df["InvoiceDate"].max())
print("Reference date:", reference_date)

Latest transaction: 2011-12-09 12:50:00
Reference date: 2011-12-10 12:50:00


In [6]:
# Step 2: Construct RFM features

rfm = df.groupby("CustomerID").agg(
    LastPurchaseDate=("InvoiceDate", "max"),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TransactionValue", "sum")
).reset_index()

rfm["Recency"] = (
    reference_date - rfm["LastPurchaseDate"]
).dt.days

rfm = rfm[
    [
        "CustomerID",
        "Recency",
        "Frequency",
        "Monetary"
    ]
]

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1437.24
3,12349.0,19,1,1457.55
4,12350.0,310,1,294.40


In [7]:
print("RFM shape:", rfm.shape)
print("Unique customers:", rfm["CustomerID"].nunique())
print("Duplicate CustomerIDs:", rfm["CustomerID"].duplicated().sum())
print("Missing values:")
print(rfm.isna().sum())

RFM shape: (4335, 4)
Unique customers: 4335
Duplicate CustomerIDs: 0
Missing values:
CustomerID    0
Recency       0
Frequency     0
Monetary      0
dtype: int64


In [8]:
rfm[["Recency", "Frequency", "Monetary"]].describe()

,Recency,Frequency,Monetary
count,4335.000000,4335.000000,4335.000000
mean,92.708651,4.248212,2017.088499
std,100.206617,7.646024,8919.392095
min,1.000000,1.000000,3.750000
25%,18.000000,1.000000,304.250000
50%,51.000000,2.000000,663.610000
75%,143.000000,5.000000,1631.475000
max,374.000000,206.000000,279138.020000


In [9]:
print("Minimum Monetary:", rfm["Monetary"].min())
print("Maximum Monetary:", rfm["Monetary"].max())
print("Customers with Monetary <= 0:",
      (rfm["Monetary"] <= 0).sum())

Minimum Monetary: 3.75
Maximum Monetary: 279138.02
Customers with Monetary <= 0: 0


In [10]:
# Reference date for Recency
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

print("Latest transaction:", df["InvoiceDate"].max())
print("Reference date:", reference_date)

Latest transaction: 2011-12-09 12:50:00
Reference date: 2011-12-10 12:50:00


In [11]:
# Construct RFM features

rfm = df.groupby("CustomerID").agg(
    LastPurchaseDate=("InvoiceDate", "max"),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TransactionValue", "sum")
).reset_index()

rfm["Recency"] = (
    reference_date - rfm["LastPurchaseDate"]
).dt.days

rfm = rfm[
    [
        "CustomerID",
        "Recency",
        "Frequency",
        "Monetary"
    ]
]

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1437.24
3,12349.0,19,1,1457.55
4,12350.0,310,1,294.40


In [12]:
print("RFM shape:", rfm.shape)
print("Unique customers:", rfm["CustomerID"].nunique())
print("Duplicate CustomerIDs:", rfm["CustomerID"].duplicated().sum())

print("\nMissing values:")
print(rfm.isna().sum())

RFM shape: (4335, 4)
Unique customers: 4335
Duplicate CustomerIDs: 0

Missing values:
CustomerID    0
Recency       0
Frequency     0
Monetary      0
dtype: int64


In [13]:
rfm[["Recency", "Frequency", "Monetary"]].describe()

,Recency,Frequency,Monetary
count,4335.000000,4335.000000,4335.000000
mean,92.708651,4.248212,2017.088499
std,100.206617,7.646024,8919.392095
min,1.000000,1.000000,3.750000
25%,18.000000,1.000000,304.250000
50%,51.000000,2.000000,663.610000
75%,143.000000,5.000000,1631.475000
max,374.000000,206.000000,279138.020000


In [14]:
# Behavioral Feature 1: First and last purchase dates

customer_dates = df.groupby("CustomerID").agg(
    FirstPurchaseDate=("InvoiceDate", "min"),
    LastPurchaseDate=("InvoiceDate", "max")
).reset_index()

customer_dates.head()

,CustomerID,FirstPurchaseDate,LastPurchaseDate
0,12346.0,2011-01-18 10:01:00,2011-01-18 10:01:00
1,12347.0,2010-12-07 14:57:00,2011-12-07 15:52:00
2,12348.0,2010-12-16 19:09:00,2011-09-25 13:13:00
3,12349.0,2011-11-21 09:51:00,2011-11-21 09:51:00
4,12350.0,2011-02-02 16:01:00,2011-02-02 16:01:00


In [15]:
print("Customers:", len(customer_dates))
print("Missing FirstPurchaseDate:",
      customer_dates["FirstPurchaseDate"].isna().sum())
print("Missing LastPurchaseDate:",
      customer_dates["LastPurchaseDate"].isna().sum())

Customers: 4335
Missing FirstPurchaseDate: 0
Missing LastPurchaseDate: 0


In [16]:
customer_dates["CustomerLifespanDays"] = (
    customer_dates["LastPurchaseDate"] -
    customer_dates["FirstPurchaseDate"]
).dt.days

customer_dates.head()

,CustomerID,FirstPurchaseDate,LastPurchaseDate,CustomerLifespanDays
0,12346.0,2011-01-18 10:01:00,2011-01-18 10:01:00,0
1,12347.0,2010-12-07 14:57:00,2011-12-07 15:52:00,365
2,12348.0,2010-12-16 19:09:00,2011-09-25 13:13:00,282
3,12349.0,2011-11-21 09:51:00,2011-11-21 09:51:00,0
4,12350.0,2011-02-02 16:01:00,2011-02-02 16:01:00,0


In [17]:
# Behavioral Feature 2: Purchase intervals

customer_invoice_dates = (
    df[["CustomerID", "InvoiceNo", "InvoiceDate"]]
    .drop_duplicates(subset=["CustomerID", "InvoiceNo"])
    .sort_values(["CustomerID", "InvoiceDate"])
)

customer_invoice_dates.head(10)

,CustomerID,InvoiceNo,InvoiceDate
36386,12346.0,541431,2011-01-18 10:01:00
10219,12347.0,537626,2010-12-07 14:57:00
42596,12347.0,542237,2011-01-26 14:30:00
97743,12347.0,549222,2011-04-07 10:43:00
151541,12347.0,556201,2011-06-09 13:01:00
197510,12347.0,562032,2011-08-02 08:48:00
310168,12347.0,573511,2011-10-31 12:25:00
387894,12347.0,581180,2011-12-07 15:52:00
22492,12348.0,539318,2010-12-16 19:09:00
40849,12348.0,541998,2011-01-25 10:42:00


In [18]:
customer_invoice_dates["PurchaseIntervalDays"] = (
    customer_invoice_dates
    .groupby("CustomerID")["InvoiceDate"]
    .diff()
    .dt.total_seconds()
    / (24 * 60 * 60)
)

customer_invoice_dates.head(20)

,CustomerID,InvoiceNo,InvoiceDate,PurchaseIntervalDays
36386,12346.0,541431,2011-01-18 10:01:00,NaN
10219,12347.0,537626,2010-12-07 14:57:00,NaN
42596,12347.0,542237,2011-01-26 14:30:00,49.981250
97743,12347.0,549222,2011-04-07 10:43:00,70.842361
151541,12347.0,556201,2011-06-09 13:01:00,63.095833
197510,12347.0,562032,2011-08-02 08:48:00,53.824306
310168,12347.0,573511,2011-10-31 12:25:00,90.150694
387894,12347.0,581180,2011-12-07 15:52:00,37.143750
22492,12348.0,539318,2010-12-16 19:09:00,NaN
40849,12348.0,541998,2011-01-25 10:42:00,39.647917


In [19]:
average_interval = (
    customer_invoice_dates
    .groupby("CustomerID")["PurchaseIntervalDays"]
    .mean()
    .reset_index(name="AveragePurchaseIntervalDays")
)

average_interval.head()

,CustomerID,AveragePurchaseIntervalDays
0,12346.0,NaN
1,12347.0,60.839699
2,12348.0,94.250926
3,12349.0,NaN
4,12350.0,NaN


In [20]:
print("Customers:", len(average_interval))
print(
    "Customers with no previous purchase:",
    average_interval["AveragePurchaseIntervalDays"].isna().sum()
)

Customers: 4335
Customers with no previous purchase: 1505


In [21]:
average_interval["AveragePurchaseIntervalDays"].describe()

count    2830.000000
mean       73.085764
std        65.529377
min         0.000000
25%        30.079427
50%        54.025694
75%        92.697743
max       365.981944
Name: AveragePurchaseIntervalDays, dtype: float64

In [22]:
print(
    "Customers with calculated interval:",
    average_interval["AveragePurchaseIntervalDays"].notna().sum()
)

print(
    "Customers with no interval:",
    average_interval["AveragePurchaseIntervalDays"].isna().sum()
)

Customers with calculated interval: 2830
Customers with no interval: 1505


In [23]:
zero_intervals = customer_invoice_dates[
    customer_invoice_dates["PurchaseIntervalDays"] == 0
]

print("Zero-day intervals:", len(zero_intervals))

zero_intervals[
    [
        "CustomerID",
        "InvoiceNo",
        "InvoiceDate",
        "PurchaseIntervalDays"
    ]
].head(20)

Zero-day intervals: 122


,CustomerID,InvoiceNo,InvoiceDate,PurchaseIntervalDays
157270,12415.0,556918,2011-06-15 13:37:00,0.0
169139,12424.0,558537,2011-06-30 12:06:00,0.0
302527,12569.0,572895,2011-10-26 14:39:00,0.0
387948,12569.0,581184,2011-12-07 16:24:00,0.0
380614,12841.0,580601,2011-12-05 11:13:00,0.0
249565,12906.0,567883,2011-09-22 15:03:00,0.0
298058,12921.0,572452,2011-10-24 12:36:00,0.0
89383,12971.0,547946,2011-03-28 14:59:00,0.0
181764,12971.0,560183,2011-07-15 11:48:00,0.0
328000,12971.0,575390,2011-11-09 15:31:00,0.0


In [24]:
print(
    "Minimum positive interval:",
    customer_invoice_dates.loc[
        customer_invoice_dates["PurchaseIntervalDays"] > 0,
        "PurchaseIntervalDays"
    ].min()
)

Minimum positive interval: 0.0006944444444444445


In [25]:
# Behavioral Feature 3: Total quantity purchased

total_quantity = (
    df.groupby("CustomerID")["Quantity"]
    .sum()
    .reset_index(name="TotalQuantity")
)

total_quantity.head()


,CustomerID,TotalQuantity
0,12346.0,74215
1,12347.0,2458
2,12348.0,2332
3,12349.0,630
4,12350.0,196


In [26]:
print("Customers:", len(total_quantity))
print("Missing values:", total_quantity["TotalQuantity"].isna().sum())
print("Minimum:", total_quantity["TotalQuantity"].min())
print("Maximum:", total_quantity["TotalQuantity"].max())

Customers: 4335
Missing values: 0
Minimum: 1
Maximum: 196844


In [27]:
total_quantity["TotalQuantity"].describe()

count      4335.000000
mean       1186.143714
std        5040.407448
min           1.000000
25%         159.000000
50%         377.000000
75%         989.500000
max      196844.000000
Name: TotalQuantity, dtype: float64

In [28]:
# Behavioral Feature 4: Product diversity

product_diversity = (
    df.groupby("CustomerID")["StockCode"]
    .nunique()
    .reset_index(name="ProductDiversity")
)

product_diversity.head()

,CustomerID,ProductDiversity
0,12346.0,1
1,12347.0,103
2,12348.0,21
3,12349.0,72
4,12350.0,16


In [29]:
print("Customers:", len(product_diversity))
print("Missing values:",
      product_diversity["ProductDiversity"].isna().sum())
print("Minimum:",
      product_diversity["ProductDiversity"].min())
print("Maximum:",
      product_diversity["ProductDiversity"].max())

Customers: 4335
Missing values: 0
Minimum: 1
Maximum: 1786


In [30]:
product_diversity["ProductDiversity"].describe()

count    4335.000000
mean       61.421684
std        85.312346
min         1.000000
25%        16.000000
50%        35.000000
75%        77.000000
max      1786.000000
Name: ProductDiversity, dtype: float64

In [31]:
# Behavioral Feature 5: Average basket quantity

average_basket_quantity = (
    total_quantity
    .merge(
        rfm[["CustomerID", "Frequency"]],
        on="CustomerID",
        how="left"
    )
)

average_basket_quantity["AverageBasketQuantity"] = (
    average_basket_quantity["TotalQuantity"] /
    average_basket_quantity["Frequency"]
)

average_basket_quantity.head()

,CustomerID,TotalQuantity,Frequency,AverageBasketQuantity
0,12346.0,74215,1,74215.000000
1,12347.0,2458,7,351.142857
2,12348.0,2332,4,583.000000
3,12349.0,630,1,630.000000
4,12350.0,196,1,196.000000


In [32]:
print("Customers:", len(average_basket_quantity))

print(
    "Missing values:",
    average_basket_quantity["AverageBasketQuantity"].isna().sum()
)

print(
    "Infinite values:",
    np.isinf(
        average_basket_quantity["AverageBasketQuantity"]
    ).sum()
)

Customers: 4335
Missing values: 0
Infinite values: 0


In [33]:
average_basket_quantity[
    "AverageBasketQuantity"
].describe()

count     4335.000000
mean       255.133835
std       1316.212816
min          1.000000
25%         92.500000
50%        161.000000
75%        271.714286
max      74215.000000
Name: AverageBasketQuantity, dtype: float64

In [34]:
# Behavioral Feature 6: Average transaction value

average_transaction_value = (
    rfm[["CustomerID", "Frequency", "Monetary"]]
    .copy()
)

average_transaction_value["AverageTransactionValue"] = (
    average_transaction_value["Monetary"] /
    average_transaction_value["Frequency"]
)

average_transaction_value.head()

,CustomerID,Frequency,Monetary,AverageTransactionValue
0,12346.0,1,77183.60,77183.600000
1,12347.0,7,4310.00,615.714286
2,12348.0,4,1437.24,359.310000
3,12349.0,1,1457.55,1457.550000
4,12350.0,1,294.40,294.400000


In [35]:
print("Customers:", len(average_transaction_value))

print(
    "Missing values:",
    average_transaction_value["AverageTransactionValue"].isna().sum()
)

print(
    "Infinite values:",
    np.isinf(
        average_transaction_value["AverageTransactionValue"]
    ).sum()
)

Customers: 4335
Missing values: 0
Infinite values: 0


In [36]:
average_transaction_value[
    "AverageTransactionValue"
].describe()

count     4335.000000
mean       415.474833
std       1800.666323
min          3.750000
25%        177.088333
50%        289.652500
75%        423.366250
max      84236.250000
Name: AverageTransactionValue, dtype: float64

In [37]:
# Temporal analysis: monthly transaction summary

df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")

monthly_summary = (
    df.groupby("YearMonth")
    .agg(
        Transactions=("InvoiceNo", "nunique"),
        Quantity=("Quantity", "sum"),
        Spending=("TransactionValue", "sum"),
        Customers=("CustomerID", "nunique")
    )
    .reset_index()
)

monthly_summary

,YearMonth,Transactions,Quantity,Spending,Customers
0,2010-12,1396,310633,565764.560,885
1,2011-01,984,348183,562897.910,739
2,2011-02,992,264789,442493.590,757
3,2011-03,1312,347257,583843.850,973
4,2011-04,1140,291126,454855.881,853
5,2011-05,1545,372599,659693.490,1055
6,2011-06,1390,362736,653764.920,990
7,2011-07,1321,367064,592103.791,946
8,2011-08,1268,397106,635864.380,933
9,2011-09,1742,542847,939682.632,1261


In [38]:
print("Number of months:", len(monthly_summary))
print("First month:", monthly_summary["YearMonth"].min())
print("Last month:", monthly_summary["YearMonth"].max())

Number of months: 13
First month: 2010-12
Last month: 2011-12


In [39]:
# Define historical and recent periods

split_date = pd.Timestamp("2011-06-30 23:59:59")

df["Period"] = np.where(
    df["InvoiceDate"] <= split_date,
    "Historical",
    "Recent"
)

print(df["Period"].value_counts())

Period
Recent        221264
Historical    170034
Name: count, dtype: int64


In [40]:
period_summary = (
    df.groupby("Period")
    .agg(
        StartDate=("InvoiceDate", "min"),
        EndDate=("InvoiceDate", "max"),
        Transactions=("InvoiceNo", "nunique"),
        Customers=("CustomerID", "nunique"),
        Quantity=("Quantity", "sum"),
        Spending=("TransactionValue", "sum")
    )
    .reset_index()
)

period_summary

,Period,StartDate,EndDate,Transactions,Customers,Quantity,Spending
0,Historical,2010-12-01 08:26:00,2011-06-30 20:08:00,8759,2959,2297323,3923314.201
1,Recent,2011-07-01 08:16:00,2011-12-09 12:50:00,9657,3361,2844610,4820764.443


In [41]:
# Calculate period lengths in months

historical_months = 7
recent_months = 6

print("Historical months:", historical_months)
print("Recent months:", recent_months)

Historical months: 7
Recent months: 6


In [42]:
# Customer-level historical/recent behavior

customer_period = (
    df.groupby(["CustomerID", "Period"])
    .agg(
        Spending=("TransactionValue", "sum"),
        Frequency=("InvoiceNo", "nunique"),
        Quantity=("Quantity", "sum")
    )
    .reset_index()
)

customer_period.head()

,CustomerID,Period,Spending,Frequency,Quantity
0,12346.0,Historical,77183.60,1,74215
1,12347.0,Historical,2205.95,4,1313
2,12347.0,Recent,2104.05,3,1145
3,12348.0,Historical,1167.24,3,2116
4,12348.0,Recent,270.00,1,216


In [43]:
# Step 3: Monthly-normalized historical and recent behavior

customer_period["MonthlySpending"] = np.where(
    customer_period["Period"] == "Historical",
    customer_period["Spending"] / historical_months,
    customer_period["Spending"] / recent_months
)

customer_period["MonthlyFrequency"] = np.where(
    customer_period["Period"] == "Historical",
    customer_period["Frequency"] / historical_months,
    customer_period["Frequency"] / recent_months
)

customer_period["MonthlyQuantity"] = np.where(
    customer_period["Period"] == "Historical",
    customer_period["Quantity"] / historical_months,
    customer_period["Quantity"] / recent_months
)

customer_period.head()

,CustomerID,Period,Spending,Frequency,Quantity,MonthlySpending,MonthlyFrequency,MonthlyQuantity
0,12346.0,Historical,77183.60,1,74215,11026.228571,0.142857,10602.142857
1,12347.0,Historical,2205.95,4,1313,315.135714,0.571429,187.571429
2,12347.0,Recent,2104.05,3,1145,350.675000,0.500000,190.833333
3,12348.0,Historical,1167.24,3,2116,166.748571,0.428571,302.285714
4,12348.0,Recent,270.00,1,216,45.000000,0.166667,36.000000


In [44]:
print(
    customer_period[
        [
            "CustomerID",
            "Period",
            "MonthlySpending",
            "MonthlyFrequency",
            "MonthlyQuantity"
        ]
    ].head()
)

   CustomerID      Period  MonthlySpending  MonthlyFrequency  MonthlyQuantity
0     12346.0  Historical     11026.228571          0.142857     10602.142857
1     12347.0  Historical       315.135714          0.571429       187.571429
2     12347.0      Recent       350.675000          0.500000       190.833333
3     12348.0  Historical       166.748571          0.428571       302.285714
4     12348.0      Recent        45.000000          0.166667        36.000000


In [45]:
# Step 4: Separate historical and recent customer behavior

historical = (
    customer_period[customer_period["Period"] == "Historical"]
    [
        [
            "CustomerID",
            "MonthlySpending",
            "MonthlyFrequency",
            "MonthlyQuantity"
        ]
    ]
    .rename(columns={
        "MonthlySpending": "HistoricalMonthlySpending",
        "MonthlyFrequency": "HistoricalMonthlyFrequency",
        "MonthlyQuantity": "HistoricalMonthlyQuantity"
    })
)

recent = (
    customer_period[customer_period["Period"] == "Recent"]
    [
        [
            "CustomerID",
            "MonthlySpending",
            "MonthlyFrequency",
            "MonthlyQuantity"
        ]
    ]
    .rename(columns={
        "MonthlySpending": "RecentMonthlySpending",
        "MonthlyFrequency": "RecentMonthlyFrequency",
        "MonthlyQuantity": "RecentMonthlyQuantity"
    })
)

print("Historical customer records:", len(historical))
print("Recent customer records:", len(recent))

Historical customer records: 2959
Recent customer records: 3361


In [46]:
# Step 5: Combine historical and recent behavior

customer_trends = historical.merge(
    recent,
    on="CustomerID",
    how="outer"
)

print("Customers in trend dataset:", len(customer_trends))
print()
print("Missing historical values:")
print(customer_trends[
    [
        "HistoricalMonthlySpending",
        "HistoricalMonthlyFrequency",
        "HistoricalMonthlyQuantity"
    ]
].isna().sum())

print()
print("Missing recent values:")
print(customer_trends[
    [
        "RecentMonthlySpending",
        "RecentMonthlyFrequency",
        "RecentMonthlyQuantity"
    ]
].isna().sum())

Customers in trend dataset: 4335

Missing historical values:
HistoricalMonthlySpending     1376
HistoricalMonthlyFrequency    1376
HistoricalMonthlyQuantity     1376
dtype: int64

Missing recent values:
RecentMonthlySpending     974
RecentMonthlyFrequency    974
RecentMonthlyQuantity     974
dtype: int64


In [47]:
# Step 6: Treat absence of activity as zero activity

trend_columns = [
    "HistoricalMonthlySpending",
    "HistoricalMonthlyFrequency",
    "HistoricalMonthlyQuantity",
    "RecentMonthlySpending",
    "RecentMonthlyFrequency",
    "RecentMonthlyQuantity"
]

customer_trends[trend_columns] = (
    customer_trends[trend_columns].fillna(0)
)

print("Missing values after filling inactive periods:")
print(customer_trends[trend_columns].isna().sum())

Missing values after filling inactive periods:
HistoricalMonthlySpending     0
HistoricalMonthlyFrequency    0
HistoricalMonthlyQuantity     0
RecentMonthlySpending         0
RecentMonthlyFrequency        0
RecentMonthlyQuantity         0
dtype: int64


In [48]:
# Step 7: Calculate temporal trend features

def calculate_trend(recent, historical):
    trend = pd.Series(np.nan, index=recent.index)

    both_active = historical > 0
    both_zero = (historical == 0) & (recent == 0)

    trend[both_active] = (
        (recent[both_active] - historical[both_active])
        / historical[both_active]
    )

    trend[both_zero] = 0

    return trend


customer_trends["SpendingTrend"] = calculate_trend(
    customer_trends["RecentMonthlySpending"],
    customer_trends["HistoricalMonthlySpending"]
)

customer_trends["FrequencyTrend"] = calculate_trend(
    customer_trends["RecentMonthlyFrequency"],
    customer_trends["HistoricalMonthlyFrequency"]
)

customer_trends["QuantityTrend"] = calculate_trend(
    customer_trends["RecentMonthlyQuantity"],
    customer_trends["HistoricalMonthlyQuantity"]
)

In [49]:
customer_trends[
    [
        "CustomerID",
        "HistoricalMonthlySpending",
        "RecentMonthlySpending",
        "SpendingTrend",
        "HistoricalMonthlyFrequency",
        "RecentMonthlyFrequency",
        "FrequencyTrend",
        "HistoricalMonthlyQuantity",
        "RecentMonthlyQuantity",
        "QuantityTrend"
    ]
].head(20)

,CustomerID,HistoricalMonthlySpending,RecentMonthlySpending,SpendingTrend,HistoricalMonthlyFrequency,RecentMonthlyFrequency,FrequencyTrend,HistoricalMonthlyQuantity,RecentMonthlyQuantity,QuantityTrend
0,12346.0,11026.228571,0.000000,-1.000000,0.142857,0.000000,-1.000000,10602.142857,0.000000,-1.000000
1,12347.0,315.135714,350.675000,0.112775,0.571429,0.500000,-0.125000,187.571429,190.833333,0.017390
2,12348.0,166.748571,45.000000,-0.730133,0.428571,0.166667,-0.611111,302.285714,36.000000,-0.880907
3,12349.0,0.000000,242.925000,NaN,0.000000,0.166667,NaN,0.000000,105.000000,NaN
4,12350.0,42.057143,0.000000,-1.000000,0.142857,0.000000,-1.000000,28.000000,0.000000,-1.000000
5,12352.0,91.644286,124.038333,0.353476,0.571429,0.500000,-0.125000,35.571429,46.166667,0.297858
6,12353.0,12.714286,0.000000,-1.000000,0.142857,0.000000,-1.000000,2.857143,0.000000,-1.000000
7,12354.0,154.200000,0.000000,-1.000000,0.142857,0.000000,-1.000000,75.714286,0.000000,-1.000000
8,12355.0,65.628571,0.000000,-1.000000,0.142857,0.000000,-1.000000,34.285714,0.000000,-1.000000
9,12356.0,347.011429,9.725000,-0.971975,0.285714,0.166667,-0.416667,224.000000,0.833333,-0.996280


In [50]:
print("Missing SpendingTrend:",
      customer_trends["SpendingTrend"].isna().sum())

print("Missing FrequencyTrend:",
      customer_trends["FrequencyTrend"].isna().sum())

print("Missing QuantityTrend:",
      customer_trends["QuantityTrend"].isna().sum())

print("Infinite SpendingTrend:",
      np.isinf(customer_trends["SpendingTrend"]).sum())

Missing SpendingTrend: 1376
Missing FrequencyTrend: 1376
Missing QuantityTrend: 1376
Infinite SpendingTrend: 0


In [51]:
# Identify customers with no historical activity

customer_trends["NewCustomer"] = (
    (customer_trends["HistoricalMonthlyFrequency"] == 0) &
    (customer_trends["RecentMonthlyFrequency"] > 0)
).astype(int)

print(
    "New customers:",
    customer_trends["NewCustomer"].sum()
)

print(
    "Existing customers:",
    (customer_trends["NewCustomer"] == 0).sum()
)

New customers: 1376
Existing customers: 2959


In [52]:
# Handle undefined trends for new customers

trend_features = [
    "SpendingTrend",
    "FrequencyTrend",
    "QuantityTrend"
]

customer_trends[trend_features] = (
    customer_trends[trend_features].fillna(0)
)

print("Missing trend values after treatment:")

print(
    customer_trends[trend_features]
    .isna()
    .sum()
)

Missing trend values after treatment:
SpendingTrend     0
FrequencyTrend    0
QuantityTrend     0
dtype: int64


In [53]:
customer_trends[
    [
        "SpendingTrend",
        "FrequencyTrend",
        "QuantityTrend"
    ]
].describe()

,SpendingTrend,FrequencyTrend,QuantityTrend
count,4335.000000,4335.000000,4335.000000
mean,15.921225,0.028760,11.478920
std,1029.381003,1.125987,717.613622
min,-1.000000,-1.000000,-1.000000
25%,-0.688380,-0.611111,-0.723265
50%,0.000000,0.000000,0.000000
75%,0.139753,0.166667,0.191513
max,67774.126437,13.000000,47246.083333


In [54]:
print(
    "Infinite values:"
)

print(
    np.isinf(
        customer_trends[
            [
                "SpendingTrend",
                "FrequencyTrend",
                "QuantityTrend"
            ]
        ]
    ).sum()
)

Infinite values:
SpendingTrend     0
FrequencyTrend    0
QuantityTrend     0
dtype: int64


In [55]:
customer_trends[
    [
        "SpendingTrend",
        "FrequencyTrend",
        "QuantityTrend"
    ]
].describe()

,SpendingTrend,FrequencyTrend,QuantityTrend
count,4335.000000,4335.000000,4335.000000
mean,15.921225,0.028760,11.478920
std,1029.381003,1.125987,717.613622
min,-1.000000,-1.000000,-1.000000
25%,-0.688380,-0.611111,-0.723265
50%,0.000000,0.000000,0.000000
75%,0.139753,0.166667,0.191513
max,67774.126437,13.000000,47246.083333


In [56]:
print("Infinite values:")
print(
    np.isinf(
        customer_trends[
            [
                "SpendingTrend",
                "FrequencyTrend",
                "QuantityTrend"
            ]
        ]
    ).sum()
)

Infinite values:
SpendingTrend     0
FrequencyTrend    0
QuantityTrend     0
dtype: int64


In [57]:
# Behavioral feature matrix

behavioral_features = (
    customer_dates[
        [
            "CustomerID",
            "CustomerLifespanDays"
        ]
    ]
    .merge(
        average_interval,
        on="CustomerID",
        how="left"
    )
    .merge(
        total_quantity,
        on="CustomerID",
        how="left"
    )
    .merge(
        product_diversity,
        on="CustomerID",
        how="left"
    )
    .merge(
        average_basket_quantity[
            [
                "CustomerID",
                "AverageBasketQuantity"
            ]
        ],
        on="CustomerID",
        how="left"
    )
    .merge(
        average_transaction_value[
            [
                "CustomerID",
                "AverageTransactionValue"
            ]
        ],
        on="CustomerID",
        how="left"
    )
)

print("Behavioral matrix shape:", behavioral_features.shape)
behavioral_features.head()

Behavioral matrix shape: (4335, 7)


,CustomerID,CustomerLifespanDays,AveragePurchaseIntervalDays,TotalQuantity,ProductDiversity,AverageBasketQuantity,AverageTransactionValue
0,12346.0,0,NaN,74215,1,74215.000000,77183.600000
1,12347.0,365,60.839699,2458,103,351.142857,615.714286
2,12348.0,282,94.250926,2332,21,583.000000,359.310000
3,12349.0,0,NaN,630,72,630.000000,1457.550000
4,12350.0,0,NaN,196,16,196.000000,294.400000


In [58]:
print("Missing values:")
print(behavioral_features.isna().sum())

Missing values:
CustomerID                        0
CustomerLifespanDays              0
AveragePurchaseIntervalDays    1505
TotalQuantity                     0
ProductDiversity                  0
AverageBasketQuantity             0
AverageTransactionValue           0
dtype: int64


In [59]:
# Median purchase interval among customers with >= 2 invoices

median_purchase_interval = (
    behavioral_features["AveragePurchaseIntervalDays"]
    .median()
)

print(
    "Median purchase interval:",
    median_purchase_interval
)

Median purchase interval: 54.02569444444444


In [60]:
# Fill undefined intervals for single-invoice customers

behavioral_features["AveragePurchaseIntervalDays"] = (
    behavioral_features["AveragePurchaseIntervalDays"]
    .fillna(median_purchase_interval)
)

print(
    "Missing AveragePurchaseIntervalDays:",
    behavioral_features[
        "AveragePurchaseIntervalDays"
    ].isna().sum()
)

Missing AveragePurchaseIntervalDays: 0


In [61]:
# Step 12: Combine RFM + behavioral + temporal features

customer_features = (
    rfm
    .merge(
        behavioral_features,
        on="CustomerID",
        how="left"
    )
    .merge(
        customer_trends[
            [
                "CustomerID",
                "SpendingTrend",
                "FrequencyTrend",
                "QuantityTrend",
                "NewCustomer"
            ]
        ],
        on="CustomerID",
        how="left"
    )
)

print("Customer feature matrix shape:", customer_features.shape)
print("Unique customers:", customer_features["CustomerID"].nunique())

Customer feature matrix shape: (4335, 14)
Unique customers: 4335


In [62]:
print("\nColumns:")
for i, column in enumerate(customer_features.columns, start=1):
    print(i, "→", column)


Columns:
1 → CustomerID
2 → Recency
3 → Frequency
4 → Monetary
5 → CustomerLifespanDays
6 → AveragePurchaseIntervalDays
7 → TotalQuantity
8 → ProductDiversity
9 → AverageBasketQuantity
10 → AverageTransactionValue
11 → SpendingTrend
12 → FrequencyTrend
13 → QuantityTrend
14 → NewCustomer


In [63]:
print("\nMissing values:")
print(customer_features.isna().sum())


Missing values:
CustomerID                     0
Recency                        0
Frequency                      0
Monetary                       0
CustomerLifespanDays           0
AveragePurchaseIntervalDays    0
TotalQuantity                  0
ProductDiversity               0
AverageBasketQuantity          0
AverageTransactionValue        0
SpendingTrend                  0
FrequencyTrend                 0
QuantityTrend                  0
NewCustomer                    0
dtype: int64


In [64]:
print(
    "\nDuplicate CustomerIDs:",
    customer_features["CustomerID"].duplicated().sum()
)


Duplicate CustomerIDs: 0


In [65]:
numeric_features = customer_features.select_dtypes(
    include=np.number
).columns

print(
    "\nInfinite values:"
)

print(
    np.isinf(
        customer_features[numeric_features]
    ).sum()
)


Infinite values:
CustomerID                     0
Recency                        0
Frequency                      0
Monetary                       0
CustomerLifespanDays           0
AveragePurchaseIntervalDays    0
TotalQuantity                  0
ProductDiversity               0
AverageBasketQuantity          0
AverageTransactionValue        0
SpendingTrend                  0
FrequencyTrend                 0
QuantityTrend                  0
NewCustomer                    0
dtype: int64


In [66]:
# Step 17: Save customer-level feature matrix

FEATURE_PATH = Path("../data/processed/customer_features.csv")

customer_features.to_csv(
    FEATURE_PATH,
    index=False
)

print("Saved customer feature matrix to:")
print(FEATURE_PATH)

print("Shape:", customer_features.shape)

Saved customer feature matrix to:
..\data\processed\customer_features.csv
Shape: (4335, 14)


In [67]:
# Verify saved customer feature matrix

verified_features = pd.read_csv(FEATURE_PATH)

print("Saved shape:", verified_features.shape)
print("Unique customers:", verified_features["CustomerID"].nunique())
print("Duplicate CustomerIDs:",
      verified_features["CustomerID"].duplicated().sum())
print("Missing values:", verified_features.isna().sum().sum())

Saved shape: (4335, 14)
Unique customers: 4335
Duplicate CustomerIDs: 0
Missing values: 0


In [68]:
# Step 1: Define modeling features

feature_columns = [
    "Recency",
    "Frequency",
    "Monetary",
    "CustomerLifespanDays",
    "AveragePurchaseIntervalDays",
    "TotalQuantity",
    "ProductDiversity",
    "AverageBasketQuantity",
    "AverageTransactionValue",
    "SpendingTrend",
    "FrequencyTrend",
    "QuantityTrend",
    "NewCustomer"
]

X = customer_features[feature_columns].copy()

print("Number of modeling features:", len(feature_columns))
print("Feature matrix shape:", X.shape)

Number of modeling features: 13
Feature matrix shape: (4335, 13)


In [69]:
# Step 2: Feature variance

variance_table = (
    X.var()
    .sort_values()
    .to_frame(name="Variance")
)

variance_table

,Variance
NewCustomer,2.167132e-01
FrequencyTrend,1.267847e+00
Frequency,5.846168e+01
AveragePurchaseIntervalDays,2.885311e+03
ProductDiversity,7.278196e+03
Recency,1.004137e+04
CustomerLifespanDays,1.744802e+04
QuantityTrend,5.149693e+05
SpendingTrend,1.059625e+06
AverageBasketQuantity,1.732416e+06


In [70]:
# Step 3: Correlation analysis

correlation_matrix = X.corr()

correlation_matrix.round(3)

,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,TotalQuantity,ProductDiversity,AverageBasketQuantity,AverageTransactionValue,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer
Recency,1.000,-0.261,-0.121,-0.514,-0.091,-0.123,-0.300,0.009,0.001,-0.015,-0.496,-0.015,-0.294
Frequency,-0.261,1.000,0.550,0.478,-0.208,0.556,0.692,0.010,0.018,-0.004,0.148,-0.004,-0.219
Monetary,-0.121,0.550,1.000,0.225,-0.102,0.924,0.380,0.313,0.395,0.284,0.070,0.284,-0.102
CustomerLifespanDays,-0.514,0.478,0.225,1.000,0.292,0.228,0.458,-0.001,0.008,0.009,0.338,0.010,-0.556
AveragePurchaseIntervalDays,-0.091,-0.208,-0.102,0.292,1.000,-0.104,-0.136,0.008,0.014,0.039,0.023,0.040,-0.251
TotalQuantity,-0.123,0.556,0.924,0.228,-0.104,1.000,0.409,0.385,0.407,0.241,0.067,0.241,-0.104
ProductDiversity,-0.300,0.692,0.380,0.458,-0.136,0.409,1.000,0.026,0.035,-0.010,0.183,-0.010,-0.178
AverageBasketQuantity,0.009,0.010,0.313,-0.001,0.008,0.385,0.026,1.000,0.931,0.464,-0.001,0.464,-0.006
AverageTransactionValue,0.001,0.018,0.395,0.008,0.014,0.407,0.035,0.931,1.000,0.707,-0.000,0.707,-0.014
SpendingTrend,-0.015,-0.004,0.284,0.009,0.039,0.241,-0.010,0.464,0.707,1.000,0.004,1.000,-0.011


In [71]:
# Identify strongly correlated feature pairs

correlation_pairs = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i + 1, len(correlation_matrix.columns)):
        
        feature_1 = correlation_matrix.columns[i]
        feature_2 = correlation_matrix.columns[j]
        correlation = correlation_matrix.iloc[i, j]
        
        correlation_pairs.append(
            {
                "Feature 1": feature_1,
                "Feature 2": feature_2,
                "Correlation": correlation,
                "Absolute Correlation": abs(correlation)
            }
        )

correlation_pairs = pd.DataFrame(correlation_pairs)

strong_correlations = (
    correlation_pairs[
        correlation_pairs["Absolute Correlation"] >= 0.70
    ]
    .sort_values("Absolute Correlation", ascending=False)
)

strong_correlations

,Feature 1,Feature 2,Correlation,Absolute Correlation
73,SpendingTrend,QuantityTrend,0.999971,0.999971
63,AverageBasketQuantity,AverageTransactionValue,0.931467,0.931467
25,Monetary,TotalQuantity,0.923608,0.923608
68,AverageTransactionValue,SpendingTrend,0.707178,0.707178
70,AverageTransactionValue,QuantityTrend,0.707097,0.707097


In [72]:
trend_comparison = customer_features[
    ["SpendingTrend", "QuantityTrend"]
].copy()

trend_comparison["Difference"] = (
    trend_comparison["SpendingTrend"]
    - trend_comparison["QuantityTrend"]
)

print(
    "Maximum absolute difference:",
    trend_comparison["Difference"].abs().max()
)

print(
    "Mean absolute difference:",
    trend_comparison["Difference"].abs().mean()
)

print(
    "Number of exactly identical values:",
    (
        trend_comparison["Difference"] == 0
    ).sum()
)

print("\nSample:")
print(trend_comparison.head(10))

Maximum absolute difference: 20528.043103448268
Mean absolute difference: 5.247408462964988
Number of exactly identical values: 2352

Sample:
   SpendingTrend  QuantityTrend  Difference
0      -1.000000      -1.000000    0.000000
1       0.112775       0.017390    0.095384
2      -0.730133      -0.880907    0.150775
3       0.000000       0.000000    0.000000
4      -1.000000      -1.000000    0.000000
5       0.353476       0.297858    0.055618
6      -1.000000      -1.000000    0.000000
7      -1.000000      -1.000000    0.000000
8      -1.000000      -1.000000    0.000000
9      -0.971975      -0.996280    0.024305


In [73]:
customer_trends[
    [
        "CustomerID",
        "HistoricalMonthlySpending",
        "RecentMonthlySpending",
        "HistoricalMonthlyQuantity",
        "RecentMonthlyQuantity",
        "SpendingTrend",
        "QuantityTrend"
    ]
].head(20)

,CustomerID,HistoricalMonthlySpending,RecentMonthlySpending,HistoricalMonthlyQuantity,RecentMonthlyQuantity,SpendingTrend,QuantityTrend
0,12346.0,11026.228571,0.000000,10602.142857,0.000000,-1.000000,-1.000000
1,12347.0,315.135714,350.675000,187.571429,190.833333,0.112775,0.017390
2,12348.0,166.748571,45.000000,302.285714,36.000000,-0.730133,-0.880907
3,12349.0,0.000000,242.925000,0.000000,105.000000,0.000000,0.000000
4,12350.0,42.057143,0.000000,28.000000,0.000000,-1.000000,-1.000000
5,12352.0,91.644286,124.038333,35.571429,46.166667,0.353476,0.297858
6,12353.0,12.714286,0.000000,2.857143,0.000000,-1.000000,-1.000000
7,12354.0,154.200000,0.000000,75.714286,0.000000,-1.000000,-1.000000
8,12355.0,65.628571,0.000000,34.285714,0.000000,-1.000000,-1.000000
9,12356.0,347.011429,9.725000,224.000000,0.833333,-0.971975,-0.996280


In [74]:
# Compare Monetary and TotalQuantity

monetary_quantity = customer_features[
    ["Monetary", "TotalQuantity"]
].copy()

monetary_quantity["Difference"] = (
    monetary_quantity["Monetary"]
    - monetary_quantity["TotalQuantity"]
)

print("Correlation:",
      monetary_quantity["Monetary"].corr(
          monetary_quantity["TotalQuantity"]
      ))

print("\nSample:")
print(monetary_quantity.head(10))

Correlation: 0.9236081704838466

Sample:
   Monetary  TotalQuantity  Difference
0  77183.60          74215     2968.60
1   4310.00           2458     1852.00
2   1437.24           2332     -894.76
3   1457.55            630      827.55
4    294.40            196       98.40
5   1385.74            526      859.74
6     89.00             20       69.00
7   1079.40            530      549.40
8    459.40            240      219.40
9   2487.43           1573      914.43


In [75]:
# Compare AverageBasketQuantity and AverageTransactionValue

basket_value = customer_features[
    [
        "AverageBasketQuantity",
        "AverageTransactionValue"
    ]
].copy()

print(
    "Correlation:",
    basket_value["AverageBasketQuantity"].corr(
        basket_value["AverageTransactionValue"]
    )
)

print("\nSample:")
print(basket_value.head(10))

Correlation: 0.9314669254572452

Sample:
   AverageBasketQuantity  AverageTransactionValue
0           74215.000000             77183.600000
1             351.142857               615.714286
2             583.000000               359.310000
3             630.000000              1457.550000
4             196.000000               294.400000
5              75.142857               197.962857
6              20.000000                89.000000
7             530.000000              1079.400000
8             240.000000               459.400000
9             524.333333               829.143333


In [76]:
# Step 4: Check feature skewness

skewness_table = (
    X.skew()
    .sort_values(ascending=False)
    .to_frame(name="Skewness")
)

skewness_table

,Skewness
SpendingTrend,65.836811
QuantityTrend,65.829490
AverageBasketQuantity,47.672782
AverageTransactionValue,41.461617
TotalQuantity,20.415327
Monetary,19.553846
Frequency,11.969891
ProductDiversity,6.919730
FrequencyTrend,3.766761
AveragePurchaseIntervalDays,2.619739


In [77]:
# Identify highly skewed features

highly_skewed = skewness_table[
    skewness_table["Skewness"].abs() > 1
]

print("Features with |skewness| > 1:")
print(highly_skewed)

Features with |skewness| > 1:
                              Skewness
SpendingTrend                65.836811
QuantityTrend                65.829490
AverageBasketQuantity        47.672782
AverageTransactionValue      41.461617
TotalQuantity                20.415327
Monetary                     19.553846
Frequency                    11.969891
ProductDiversity              6.919730
FrequencyTrend                3.766761
AveragePurchaseIntervalDays   2.619739
Recency                       1.243824


In [78]:
# Step 5: Create a copy for transformation

X_transformed = X.copy()

print("Original shape:", X.shape)
print("Transformation matrix shape:", X_transformed.shape)

Original shape: (4335, 13)
Transformation matrix shape: (4335, 13)


In [79]:
# Step 6: Apply transformations to reduce skewness

positive_skewed_features = [
    "Frequency",
    "Monetary",
    "TotalQuantity",
    "ProductDiversity",
    "AveragePurchaseIntervalDays",
    "AverageBasketQuantity",
    "AverageTransactionValue",
    "Recency"
]

trend_features = [
    "SpendingTrend",
    "FrequencyTrend",
    "QuantityTrend"
]

# Log transformation for non-negative features
for feature in positive_skewed_features:
    X_transformed[feature] = np.log1p(
        X_transformed[feature]
    )

# Signed log transformation for trend features
for feature in trend_features:
    X_transformed[feature] = (
        np.sign(X_transformed[feature])
        * np.log1p(np.abs(X_transformed[feature]))
    )

In [80]:
# Step 7: Compare skewness before and after transformation

skewness_comparison = pd.DataFrame({
    "Before": X.skew(),
    "After": X_transformed.skew()
})

skewness_comparison[
    "Absolute Before"
] = skewness_comparison["Before"].abs()

skewness_comparison[
    "Absolute After"
] = skewness_comparison["After"].abs()

skewness_comparison.sort_values(
    "Absolute After",
    ascending=False
)

,Before,After,Absolute Before,Absolute After
SpendingTrend,65.836811,2.624291,65.836811,2.624291
QuantityTrend,65.829490,2.504201,65.829490,2.504201
AveragePurchaseIntervalDays,2.619739,-1.586408,2.619739,1.586408
Frequency,11.969891,1.213950,11.969891,1.213950
FrequencyTrend,3.766761,1.136982,3.766761,1.136982
NewCustomer,0.784783,0.784783,0.784783,0.784783
CustomerLifespanDays,0.456079,0.456079,0.456079,0.456079
Monetary,19.553846,0.395433,19.553846,0.395433
Recency,1.243824,-0.378946,1.243824,0.378946
AverageBasketQuantity,47.672782,-0.274855,47.672782,0.274855


In [81]:
# Step 8: Validate transformed feature matrix

print("Shape:", X_transformed.shape)

print("\nMissing values:")
print(X_transformed.isna().sum())

print("\nInfinite values:")
print(
    np.isinf(X_transformed).sum()
)

Shape: (4335, 13)

Missing values:
Recency                        0
Frequency                      0
Monetary                       0
CustomerLifespanDays           0
AveragePurchaseIntervalDays    0
TotalQuantity                  0
ProductDiversity               0
AverageBasketQuantity          0
AverageTransactionValue        0
SpendingTrend                  0
FrequencyTrend                 0
QuantityTrend                  0
NewCustomer                    0
dtype: int64

Infinite values:
Recency                        0
Frequency                      0
Monetary                       0
CustomerLifespanDays           0
AveragePurchaseIntervalDays    0
TotalQuantity                  0
ProductDiversity               0
AverageBasketQuantity          0
AverageTransactionValue        0
SpendingTrend                  0
FrequencyTrend                 0
QuantityTrend                  0
NewCustomer                    0
dtype: int64


In [82]:
# Step 9: Inspect transformed feature ranges

transformed_summary = X_transformed.describe().T[
    ["min", "25%", "50%", "75%", "max"]
]

transformed_summary

,min,25%,50%,75%,max
Recency,0.693147,2.944439,3.951244,4.969813,5.926926
Frequency,0.693147,0.693147,1.098612,1.791759,5.332719
Monetary,1.558145,5.721131,6.499200,7.397853,12.539465
CustomerLifespanDays,0.000000,0.000000,92.000000,252.000000,373.000000
AveragePurchaseIntervalDays,0.000000,3.758939,4.007800,4.242177,5.905313
TotalQuantity,0.693147,5.075174,5.934894,6.898210,12.190172
ProductDiversity,0.693147,2.833213,3.583519,4.356709,7.488294
AverageBasketQuantity,0.693147,4.537961,5.087596,5.608424,11.214735
AverageTransactionValue,1.558145,5.182280,5.672128,6.050597,11.341393
SpendingTrend,-0.693147,-0.523769,0.000000,0.130811,11.123951


In [83]:
from sklearn.preprocessing import StandardScaler

# Step 10: Standardize transformed features

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_transformed)

print("Scaled matrix shape:", X_scaled.shape)

Scaled matrix shape: (4335, 13)


In [84]:
# Step 11: Convert scaled features to DataFrame

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=feature_columns,
    index=customer_features["CustomerID"]
)

X_scaled_df.head()

,Recency,Frequency,Monetary,CustomerLifespanDays,AveragePurchaseIntervalDays,TotalQuantity,ProductDiversity,AverageBasketQuantity,AverageTransactionValue,SpendingTrend,FrequencyTrend,QuantityTrend,NewCustomer
CustomerID,,,,,,,,,,,,,
12346.0,1.460165,-0.951420,3.726618,-0.986668,0.075075,3.834504,-2.521866,6.855079,7.543976,-1.060315,-1.197412,-1.008523,-0.681925
12347.0,-2.037990,1.081712,1.428561,1.776898,0.218145,1.337240,0.966846,0.907567,1.056327,0.169874,-0.144020,-0.026712,-0.681925
12348.0,0.372077,0.392407,0.554153,1.148471,0.747516,1.298687,-0.404668,1.469855,0.334369,-0.837423,-0.801545,-0.923673,-0.681925
12349.0,-0.623381,-0.951420,0.565323,-0.986668,0.075075,0.340302,0.654346,1.555893,2.212650,0.005557,0.071620,-0.050540,1.466437
12350.0,1.422757,-0.951420,-0.706655,-0.986668,0.075075,-0.512905,-0.632316,0.261948,0.067539,-1.060315,-1.197412,-1.008523,-0.681925


In [85]:
# Step 12: Check mean and standard deviation after scaling

scaling_check = pd.DataFrame({
    "Mean": X_scaled_df.mean(),
    "Std": X_scaled_df.std()
})

scaling_check

,Mean,Std
Recency,1.770210e-16,1.000115
Frequency,5.163113e-17,1.000115
Monetary,-6.294081e-16,1.000115
CustomerLifespanDays,1.147359e-17,1.000115
AveragePurchaseIntervalDays,-1.245704e-16,1.000115
TotalQuantity,5.245067e-17,1.000115
ProductDiversity,6.310472e-16,1.000115
AverageBasketQuantity,9.178868e-17,1.000115
AverageTransactionValue,-9.900065e-16,1.000115
SpendingTrend,1.721038e-17,1.000115


In [86]:
print("Missing values:", X_scaled_df.isna().sum().sum())

print(
    "Infinite values:",
    np.isinf(X_scaled_df).sum().sum()
)

Missing values: 0
Infinite values: 0


In [87]:
# Save transformed customer features

transformed_features = customer_features[
    ["CustomerID"]
].copy()

transformed_features[feature_columns] = X_transformed

TRANSFORMED_PATH = Path(
    "../data/processed/customer_features_transformed.csv"
)

transformed_features.to_csv(
    TRANSFORMED_PATH,
    index=False
)

print("Saved transformed features:")
print(TRANSFORMED_PATH)
print("Shape:", transformed_features.shape)

Saved transformed features:
..\data\processed\customer_features_transformed.csv
Shape: (4335, 14)


In [88]:
# Save scaled customer features

scaled_features = pd.DataFrame(
    X_scaled,
    columns=feature_columns
)

scaled_features.insert(
    0,
    "CustomerID",
    customer_features["CustomerID"].values
)

SCALED_PATH = Path(
    "../data/processed/customer_features_scaled.csv"
)

scaled_features.to_csv(
    SCALED_PATH,
    index=False
)

print("Saved scaled features:")
print(SCALED_PATH)
print("Shape:", scaled_features.shape)

Saved scaled features:
..\data\processed\customer_features_scaled.csv
Shape: (4335, 14)


In [89]:
# Final Notebook 3 verification

verified_scaled = pd.read_csv(SCALED_PATH)

print("Shape:", verified_scaled.shape)
print("Customers:",
      verified_scaled["CustomerID"].nunique())

print("Duplicate CustomerIDs:",
      verified_scaled["CustomerID"].duplicated().sum())

print("Missing values:",
      verified_scaled.isna().sum().sum())

print(
    "Infinite values:",
    np.isinf(
        verified_scaled.drop(columns=["CustomerID"])
    ).sum().sum()
)

Shape: (4335, 14)
Customers: 4335
Duplicate CustomerIDs: 0
Missing values: 0
Infinite values: 0
